In [9]:
import json, os, glob, sys, re
from collections import defaultdict

In [10]:
# input video
video_filename = "videos/stone_floor.mp4"

# Use Qwen2 to generate questions and scene graph based on the prompt
from qwen2 import generate_tifa_questions
prompt = "an ancient aging tiled stone floor with grass growing between the tiles"
print("Generating questions and scene graph with Qwen2...")
tifa_questions = generate_tifa_questions(prompt)

# Save the generated questions for review
with open("sample_questions.json", "w") as f:
    json.dump(tifa_questions, f, indent=4)
# print(json.dumps(generate_tifa_questions(prompt), indent=4))

print("Questions generated successfully!")

# working directory to save all the frames and corresponding answers
working_folder = "sample_results"

Generating questions and scene graph with Qwen2...


/home/liams/miniconda3/envs/eval3d/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/liams/miniconda3/envs/eval3d/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/liams/miniconda3/envs/eval3d/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Questions generated successfully!


In [11]:
# extract frames from the video
from utils import extract_frames

frames = extract_frames(video_filename, working_folder,n_div=12)

In [12]:
# VQA for all questions
# replace get_vqa_answers to use other VLMs
from qwen2 import get_vqa_answers

def get_vqa_answers_for_dir(image_directory, tifa_questions):
    
    images = glob.glob(os.path.join(image_directory, "*.png"))
    
    this_prompt_questions = tifa_questions["questions"]
    
    this_prompt_answers= {"questions": this_prompt_questions,
                          "answers": defaultdict(dict),
                          "scene_graph": tifa_questions["scene_graph"],
                          "dependencies": tifa_questions["question_dependencies"]
                          }
    
    for i, image in enumerate(images):
        print(f"Processing image {i+1}/{len(images)}: {os.path.basename(image)}")
        answers = get_vqa_answers(image, this_prompt_questions)
        this_image = os.path.basename(image)
        
        for qid, answer in answers.items():
            this_prompt_answers["answers"][qid][this_image] = answer
            
        for qid in this_prompt_questions.keys():
            if int(qid) not in answers:
                this_prompt_answers["answers"][int(qid)][this_image] = "N/A"
            
    with open(os.path.join(image_directory, "vqa_answers.json"), "w") as f:
        json.dump(this_prompt_answers, f, indent=4)

In [13]:
# Run the VQA (may take a while)
get_vqa_answers_for_dir(working_folder, tifa_questions)

Processing image 1/12: 10.png
Processing image 2/12: 5.png
Processing image 3/12: 7.png
Processing image 4/12: 9.png
Processing image 5/12: 0.png
Processing image 6/12: 2.png
Processing image 7/12: 6.png
Processing image 8/12: 3.png
Processing image 9/12: 8.png
Processing image 10/12: 1.png
Processing image 11/12: 4.png
Processing image 12/12: 11.png


In [14]:
# Compute the alignment score for the model. Also, get the score for each question and record it.

def process_directory_tifa(directory, json_name="vqa_answers.json", n_div=12):
    tifa_score_dict = json.load(open(os.path.join(directory, json_name)))
    per_question_score_dict = {}
    
    q_ids = list(tifa_score_dict["answers"].keys())
    
    question_dependencies = tifa_score_dict["dependencies"]
    
    for q_id in q_ids:
        this_scores = []

        # first, check dependency
        dependency_ids = question_dependencies[q_id]
        question_valid_flag = True
        for dep_id in dependency_ids:
            dep_id = str(dep_id)
            if dep_id in per_question_score_dict:
                if per_question_score_dict[dep_id] < 0.5:
                    question_valid_flag = False
                    break
        
        # if prerequisite question got the wrong answer, then this question is also wrong
        if not question_valid_flag:
            per_question_score_dict[q_id] = 0
            continue
        
        for i in range(n_div):
            try:
                answer = tifa_score_dict["answers"][q_id][f"{i}.png"]
            except:
                answer = "N/A"
            
            if "yes" in answer.strip().lower():
                this_scores.append(1)
            else:
                this_scores.append(0)
        
        # now pooling is here
        looped_scores = this_scores + this_scores + this_scores
        
        pooled_scores = []
        for i in range(n_div, 2*n_div):
            if looped_scores[i] == 1 and looped_scores[i+1] == 1 and looped_scores[i-1] == 1:
                pooled_scores.append(1)
            else:
                pooled_scores.append(0)
                
        per_question_score_dict[q_id] = sum(pooled_scores) / len(pooled_scores)
        
    # record per_question_scores
    tifa_score_dict["alignment_scores_per_question"] = per_question_score_dict
    
    # return average score
    avg_score =  sum(per_question_score_dict.values())/len(per_question_score_dict)
    tifa_score_dict["alignment_score"] = avg_score
    
    with open(os.path.join(directory, json_name), "w") as f:
        json.dump(tifa_score_dict, f, indent=2)
        
    return avg_score

In [15]:
# compute the final alignment score

process_directory_tifa(working_folder)

0.3333333333333333